# Phase 2: Baseline models

**Project:** SynText-Scanner  
**Goal:** Clean raw text data using NLP techniques (Lemmatization) and explore statistical differences between AI and Human-authored content.

In [7]:
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report
from xgboost import XGBClassifier
from scipy.sparse import hstack
import numpy as np

### Step 1: Data Loading & TF-IDF Vectorization

In this step, I load the pre-processed datasets and convert the cleaned text into a numerical format, preparing it for the machine learning models. 

**Key Operations:**
* **Data Loading & Sanitization:** The cleaned datasets from Phase 1 are imported. A strict `.dropna()` is applied to remove any empty strings or `NaN` values that might have been generated during the text cleaning phase.
* **Feature & Target Definition:** The `cleaned_text` is isolated as the input feature (X) and the `label` as the target variable (y).
* **TF-IDF Vectorization:** `TfidfVectorizer` is utilized to transform the text into a sparse matrix of TF-IDF features. The vocabulary is limited to `max_features=5000` to capture the most informative terms while managing computational complexity. To prevent data leakage, `.fit_transform()` is strictly applied to the training set, while only `.transform()` is used on the test set.

In [8]:
df_train = pd.read_csv('../data/processed data/train_cleaned.csv', index_col=0)
df_test = pd.read_csv('../data/processed data/test_cleaned.csv')

df_train = df_train.dropna(subset=['cleaned_text', 'label']).reset_index(drop=True)
df_test = df_test.dropna(subset=['cleaned_text', 'label']).reset_index(drop=True)

X_train_raw = df_train['cleaned_text']
y_train = df_train['label'].astype(int)
X_test_raw = df_test['cleaned_text']
y_test = df_test['label'].astype(int)

print(f'Training samples: {len(X_train_raw)}, Test samples: {len(X_test_raw)}')

vectorizer = TfidfVectorizer(max_features=5000)

X_train_tfidf = vectorizer.fit_transform(X_train_raw)
X_test_tfidf = vectorizer.transform(X_test_raw)

print(f'TF-IDF feature matrix shape: {X_train_tfidf.shape}')

Training samples: 6069, Test samples: 1367
TF-IDF feature matrix shape: (6069, 5000)


### Step 2: Baseline Model Training (Logistic Regression)

In this step, I establish the primary baseline using a linear model (Logistic Regression) to test whether simple term frequency patterns (TF-IDF) can effectively distinguish between human and AI-authored texts.

**Key Operations:**
* **Model Initialization:** A `LogisticRegression` classifier is initialized. The `random_state` is fixed for reproducibility, and `max_iter=1000` is set to ensure the optimization algorithm fully converges.
* **Training & Prediction:** The model is trained on the high-dimensional TF-IDF feature matrix and subsequently evaluated on the completely unseen 2025 test dataset.
* **Evaluation:** Performance is measured using overall accuracy and a comprehensive classification report (Precision, Recall, F1-score).

**Critical Observation:**
The model achieves an accuracy of approximately **48.7%**, which is statistically equivalent to a random coin flip. This critical finding proves that a traditional "bag-of-words" approach is fundamentally insufficient for this task. It suggests that modern AI models (in the 2025 dataset) have evolved to utilize vocabulary distributions that are virtually indistinguishable from human writing.

In [9]:
lr_model = LogisticRegression(random_state=42, max_iter=1000)
print('Training Logistic Regression model...')
lr_model.fit(X_train_tfidf, y_train)

print('Evaluating model on test set that never met...')
y_pred_lr = lr_model.predict(X_test_tfidf)

acc_lr = accuracy_score(y_test, y_pred_lr)
print(f'Logstic Regression Accuracy: {acc_lr:.4f}')

print("Classification Report")
print(classification_report(y_test, y_pred_lr, target_names=['Human (0)', 'AI (1)']))

Training Logistic Regression model...
Evaluating model on test set that never met...
Logstic Regression Accuracy: 0.4872
Classification Report
              precision    recall  f1-score   support

   Human (0)       0.49      0.65      0.56       684
      AI (1)       0.48      0.32      0.39       683

    accuracy                           0.49      1367
   macro avg       0.49      0.49      0.47      1367
weighted avg       0.49      0.49      0.47      1367



### Step 3: Non-Linear Baseline Validation (XGBoost)

To ensure the previous failure was not simply due to the linear nature of Logistic Regression, I implement an XGBoost classifier. This advanced, non-linear tree-based model is highly capable of capturing complex interactions between TF-IDF term frequencies.

**Key Operations:**
* **Hyperparameter Tuning:** The `XGBClassifier` is configured with `n_estimators=100` and `max_depth=6` to balance learning capacity and computational cost. Crucially, `tree_method='hist'` is applied to significantly accelerate training on the large, sparse TF-IDF matrix.
* **Execution:** The model is trained on the same TF-IDF feature space and evaluated against the 2025 test set.

**Critical Observation:**
Despite utilizing a vastly more powerful algorithm, the accuracy remains stagnant at exactly **50.18%**. This solidifies the core hypothesis: the limitation lies not in the choice of the machine learning algorithm, but in the feature representation itself. Pure term frequency analysis (TF-IDF) is inherently blind to the semantic nuances required to detect modern, highly capable LLMs.

In [10]:
xgb_model = XGBClassifier(random_state=42,
                          n_estimators=100,
                          max_depth=6,
                          learning_rate=0.1,
                          tree_method='hist')

xgb_model.fit(X_train_tfidf, y_train)

print('Evaluating XGBoost model on test set...')
y_pred_xgb = xgb_model.predict(X_test_tfidf)

acc_xgb = accuracy_score(y_test, y_pred_xgb)
print(f'XGBoost Accuracy: {acc_xgb:.4f}')

print("Classification Report")
print(classification_report(y_test, y_pred_xgb, target_names=['Human (0)', 'AI (1)']))

Evaluating XGBoost model on test set...
XGBoost Accuracy: 0.5018
Classification Report
              precision    recall  f1-score   support

   Human (0)       0.50      0.54      0.52       684
      AI (1)       0.50      0.47      0.48       683

    accuracy                           0.50      1367
   macro avg       0.50      0.50      0.50      1367
weighted avg       0.50      0.50      0.50      1367



### Step 4: Feature Fusion & The "Length" Hypothesis

During the Exploratory Data Analysis (EDA) in Phase 1, I discovered a significant statistical anomaly: older AI-generated texts exhibited a distinct, narrow peak in document length (`word_count`). In this final baseline experiment, I test the hypothesis that injecting this length feature into the model can break the accuracy bottleneck.

**Key Operations:**
* **Feature Concatenation:** Using `scipy.sparse.hstack`, I fuse the 5000-dimensional sparse TF-IDF matrix with the 1-dimensional dense `word_count` array, creating a combined feature space of 5001 dimensions.
* **Model Retraining:** The XGBoost classifier is retrained on this enriched dataset to see if the physical length of the text provides a decisive signal alongside term frequency.

**Critical Observation (Data Drift Revealed):**
Counterintuitively, the overall accuracy slightly decreases to **49.01%**. However, the Classification Report reveals a fascinating phenomenon:
* The Recall for Human (0) surges to **0.78**.
* The Recall for AI (1) plummets to **0.20**.

**Conclusion:** The model learned a flawed heuristic from the training data ("long text = human"). When faced with the 2025 test dataset, where modern AI effortlessly generates lengthy and highly articulated content, the model incorrectly classifies almost all long AI texts as human. This perfectly illustrates **Data Drift** and decisively concludes that traditional statistical features and bag-of-words models are obsolete for detecting state-of-the-art LLMs. Advanced semantic understanding (Deep Learning/Transformers) is strictly required.

In [11]:
train_length = df_train[['word_count']].values
test_length = df_test[['word_count']].values

X_train_combined = hstack([X_train_tfidf, train_length])
X_test_combined = hstack([X_test_tfidf, test_length])
print(f'Combined feature matrix shape: {X_train_combined.shape}')

print('Training XGBoost model with combined features...')
xgb_model_combined = XGBClassifier(random_state=42,
                                   n_estimators=100,
                                   max_depth=6,
                                   learning_rate=0.1,
                                   tree_method='hist')
xgb_model_combined.fit(X_train_combined, y_train)

print('Evaluating combined XGBoost model on test set...')
y_pred_combined = xgb_model_combined.predict(X_test_combined)
acc_combined = accuracy_score(y_test, y_pred_combined)
print(f'Combined XGBoost Accuracy: {acc_combined:.4f}')

print("Classification Report")
print(classification_report(y_test, y_pred_combined, target_names=['Human (0)', 'AI (1)']))

Combined feature matrix shape: (6069, 5001)
Training XGBoost model with combined features...
Evaluating combined XGBoost model on test set...
Combined XGBoost Accuracy: 0.4901
Classification Report
              precision    recall  f1-score   support

   Human (0)       0.49      0.78      0.61       684
      AI (1)       0.48      0.20      0.28       683

    accuracy                           0.49      1367
   macro avg       0.48      0.49      0.44      1367
weighted avg       0.48      0.49      0.44      1367

